# Qwen3 ES Review Fine-Tuning

この notebook は `career_compass/ml/es_review_qwen` を Colab GPU で実行する前提です。

- dataset repo: `saoki0913/career-compass-qwen3-es-review-data`
- adapter repo: `saoki0913/career-compass-qwen3-es-review-lora`
- base model: `Qwen/Qwen3-14B`

Colab の `Secrets` に `HF_TOKEN` を設定してから実行してください。

In [ ]:
DATASET_REPO_ID = "saoki0913/career-compass-qwen3-es-review-data"
ADAPTER_REPO_ID = "saoki0913/career-compass-qwen3-es-review-lora"
BASE_MODEL = "Qwen/Qwen3-14B"


In [ ]:
import os

from google.colab import userdata

hf_token = userdata.get("HF_TOKEN")
if not hf_token:
    raise RuntimeError("Colab Secrets に HF_TOKEN を設定してください")

os.environ["HF_TOKEN"] = hf_token
os.environ["HUGGINGFACE_TOKEN"] = hf_token
print("HF_TOKEN loaded")


In [ ]:
%cd /content
!rm -rf /content/career_compass
!git clone https://github.com/saoki0913/career_compass.git
%cd /content/career_compass
!python -m pip install -U pip
!pip install -r ml/es_review_qwen/requirements-train.txt huggingface_hub


In [ ]:
from pathlib import Path

TRAIN_SCRIPT = '''#!/usr/bin/env python3
"""Train a Qwen3 LoRA adapter for ES review with Unsloth + TRL."""

from __future__ import annotations

import argparse
import json
from pathlib import Path
from typing import Any


def _load_config(path: Path) -> dict[str, Any]:
    with path.open("r", encoding="utf-8") as file:
        return json.load(file)


def _render_chat_text(messages: list[dict[str, str]], tokenizer: Any) -> str:
    if hasattr(tokenizer, "apply_chat_template"):
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )

    chunks: list[str] = []
    for message in messages:
        role = str(message.get("role") or "user").upper()
        content = str(message.get("content") or "")
        chunks.append(f"<|{role}|>\\n{content}")
    return "\\n".join(chunks)


def main() -> None:
    parser = argparse.ArgumentParser(description="Train Qwen3 ES review LoRA with Unsloth.")
    parser.add_argument("--config", required=True, help="JSON config path")
    args = parser.parse_args()

    config_path = Path(args.config)
    config = _load_config(config_path)

    try:
        from datasets import load_dataset
        import unsloth  # noqa: F401
        from unsloth import FastLanguageModel, is_bfloat16_supported
        from trl import SFTConfig, SFTTrainer
    except ImportError as error:  # pragma: no cover - environment specific
        raise SystemExit(
            "Missing training dependencies. Run `pip install -r ml/es_review_qwen/requirements-train.txt`."
        ) from error

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=config["base_model"],
        max_seq_length=int(config.get("max_seq_length", 4096)),
        load_in_4bit=bool(config.get("load_in_4bit", True)),
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r=int(config.get("lora_rank", 32)),
        lora_alpha=int(config.get("lora_alpha", 64)),
        lora_dropout=float(config.get("lora_dropout", 0.05)),
        target_modules=list(config.get("target_modules") or []),
        bias="none",
        use_gradient_checkpointing="unsloth",
    )

    dataset = load_dataset(
        "json",
        data_files={
            "train": config["train_dataset_path"],
            "valid": config["valid_dataset_path"],
        },
    )

    def _format_row(row: dict[str, Any]) -> dict[str, str]:
        return {"text": _render_chat_text(list(row["messages"]), tokenizer)}

    train_dataset = dataset["train"].map(_format_row, remove_columns=dataset["train"].column_names)
    valid_dataset = dataset["valid"].map(_format_row, remove_columns=dataset["valid"].column_names)

    output_dir = Path(config["output_dir"])
    output_dir.mkdir(parents=True, exist_ok=True)

    bf16_enabled = bool(config.get("bf16")) if "bf16" in config else bool(is_bfloat16_supported())
    fp16_enabled = bool(config.get("fp16")) if "fp16" in config else not bf16_enabled
    training_args = {
        "output_dir": str(output_dir),
        "num_train_epochs": float(config.get("num_train_epochs", 2)),
        "per_device_train_batch_size": int(config.get("per_device_train_batch_size", 1)),
        "per_device_eval_batch_size": int(config.get("per_device_eval_batch_size", 1)),
        "gradient_accumulation_steps": int(config.get("gradient_accumulation_steps", 16)),
        "learning_rate": float(config.get("learning_rate", 2e-4)),
        "weight_decay": float(config.get("weight_decay", 0.01)),
        "logging_steps": int(config.get("logging_steps", 10)),
        "save_steps": int(config.get("save_steps", 100)),
        "eval_steps": int(config.get("eval_steps", 100)),
        "eval_strategy": "steps",
        "save_strategy": "steps",
        "lr_scheduler_type": config.get("lr_scheduler_type", "cosine"),
        "seed": int(config.get("seed", 42)),
        "report_to": [],
        "optim": config.get("optim", "paged_adamw_8bit"),
        "bf16": bf16_enabled,
        "fp16": fp16_enabled,
    }
    if config.get("warmup_steps") is not None:
        training_args["warmup_steps"] = int(config["warmup_steps"])
    elif config.get("warmup_ratio") is not None:
        training_args["warmup_ratio"] = float(config["warmup_ratio"])

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_dataset,
        eval_dataset=valid_dataset,
        dataset_text_field="text",
        max_seq_length=int(config.get("max_seq_length", 4096)),
        packing=False,
        args=SFTConfig(**training_args),
    )

    train_result = trainer.train()
    trainer.save_model(str(output_dir))
    tokenizer.save_pretrained(str(output_dir))

    summary = {
        "base_model": config["base_model"],
        "output_dir": str(output_dir),
        "train_rows": len(train_dataset),
        "valid_rows": len(valid_dataset),
        "train_runtime_seconds": getattr(train_result, "metrics", {}).get("train_runtime"),
        "train_loss": getattr(train_result, "metrics", {}).get("train_loss"),
        "bf16": bf16_enabled,
        "fp16": fp16_enabled,
        "config": config,
    }
    (output_dir / "training_summary.json").write_text(
        json.dumps(summary, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    print(json.dumps(summary, ensure_ascii=False, indent=2))


if __name__ == "__main__":
    main()
'''

CONFIG_14B = '''{
  "base_model": "Qwen/Qwen3-14B",
  "train_dataset_path": "ml/es_review_qwen/data/generated/sft/train.jsonl",
  "valid_dataset_path": "ml/es_review_qwen/data/generated/sft/valid.jsonl",
  "output_dir": "ml/es_review_qwen/outputs/qwen3-14b-es-review-lora",
  "max_seq_length": 4096,
  "load_in_4bit": true,
  "lora_rank": 32,
  "lora_alpha": 64,
  "lora_dropout": 0.0,
  "target_modules": [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj"
  ],
  "per_device_train_batch_size": 1,
  "per_device_eval_batch_size": 1,
  "gradient_accumulation_steps": 16,
  "learning_rate": 0.0002,
  "num_train_epochs": 2,
  "warmup_ratio": 0.03,
  "weight_decay": 0.01,
  "optim": "paged_adamw_8bit",
  "logging_steps": 10,
  "save_steps": 100,
  "eval_steps": 100,
  "seed": 42
}'''

CONFIG_T4 = '''{
  "base_model": "Qwen/Qwen3-8B",
  "train_dataset_path": "ml/es_review_qwen/data/generated/sft/train.jsonl",
  "valid_dataset_path": "ml/es_review_qwen/data/generated/sft/valid.jsonl",
  "output_dir": "ml/es_review_qwen/outputs/qwen3-8b-es-review-lora",
  "max_seq_length": 2048,
  "load_in_4bit": true,
  "lora_rank": 16,
  "lora_alpha": 32,
  "lora_dropout": 0.0,
  "target_modules": [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj"
  ],
  "per_device_train_batch_size": 1,
  "per_device_eval_batch_size": 1,
  "gradient_accumulation_steps": 16,
  "learning_rate": 0.0002,
  "num_train_epochs": 2,
  "warmup_ratio": 0.03,
  "weight_decay": 0.01,
  "optim": "paged_adamw_8bit",
  "logging_steps": 10,
  "save_steps": 100,
  "eval_steps": 100,
  "seed": 42
}'''

Path('/content/career_compass/ml/es_review_qwen/scripts/train_unsloth_sft.py').write_text(TRAIN_SCRIPT, encoding='utf-8')
Path('/content/career_compass/ml/es_review_qwen/configs/qwen3_14b_lora.json').write_text(CONFIG_14B, encoding='utf-8')
Path('/content/career_compass/ml/es_review_qwen/configs/qwen3_8b_t4_lora.json').write_text(CONFIG_T4, encoding='utf-8')
print('patched training files')


In [ ]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id=DATASET_REPO_ID,
    repo_type="dataset",
    local_dir="/content/career_compass/ml/es_review_qwen/data/generated",
    local_dir_use_symlinks=False,
    token=os.environ["HF_TOKEN"],
)
print("dataset downloaded")


In [ ]:
!nvidia-smi

import torch

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"
print({"gpu": gpu_name})
if "T4" in gpu_name:
    TRAIN_CONFIG = "ml/es_review_qwen/configs/qwen3_8b_t4_lora.json"
    OUTPUT_DIRNAME = "qwen3-8b-es-review-lora"
    print("T4 を検出したので 8B/T4 config を使います。")
else:
    TRAIN_CONFIG = "ml/es_review_qwen/configs/qwen3_14b_lora.json"
    OUTPUT_DIRNAME = "qwen3-14b-es-review-lora"
    print("14B config を使います。")


In [ ]:
!python ml/es_review_qwen/scripts/train_unsloth_sft.py --config {TRAIN_CONFIG}


In [ ]:
!python ml/es_review_qwen/scripts/push_artifact_to_hub.py \
  --source-dir ml/es_review_qwen/outputs/{OUTPUT_DIRNAME} \
  --repo-id saoki0913/career-compass-qwen3-es-review-lora \
  --repo-type model \
  --private


In [ ]:
!ls -lah ml/es_review_qwen/outputs/{OUTPUT_DIRNAME}
!tar -czf /content/{OUTPUT_DIRNAME}.tar.gz -C ml/es_review_qwen/outputs {OUTPUT_DIRNAME}
print({"adapter_repo": ADAPTER_REPO_ID, "backup_tar": f"/content/{OUTPUT_DIRNAME}.tar.gz"})
